In [1]:
!rm -rf TIGeR-Text-Image-Generative-Repair
!git clone https://github.com/namaray/TIGeR-Text-Image-Generative-Repair.git
%cd TIGeR-Text-Image-Generative-Repair
!git pull
# Force reinstall to bypass Kaggle's cached old package
!pip install --force-reinstall -e ".[dev,vlm,gen]" -q
# Confirm import-abo is registered
!python -m tiger.cli --help | grep import

## 1. Import ABO Dataset
Make sure you have added the `khyeh0719/amazon-berkeley-objects-small` dataset to this notebook.

In [2]:
!python -m tiger.cli import-abo \
    --listings-dir /kaggle/input/amazon-berkeley-objects-small/abo-listings/listings/metadata \
    --images-csv /kaggle/input/amazon-berkeley-objects-small/abo-images-small/images/metadata/images.csv.gz \
    --images-dir /kaggle/input/amazon-berkeley-objects-small/abo-images-small/images/small

usage: tiger [-h] [--config CONFIG] [--source SOURCE] [--caption CAPTION]
             [--output OUTPUT] [--seed SEED] [--seeds SEEDS] [--sample SAMPLE]
             [--split {report,calibration}] [--independent] [--vlm-judge]
             [--generative-fallback] [--listings-dir LISTINGS_DIR]
             [--images-csv IMAGES_CSV] [--images-dir IMAGES_DIR]
             {synthgen,import-fashion,noise,calibrate,detect,evaluate,analyze,train-arbiter,route,repair,sweep,calibrate-fusion,ablate,ablate-repair,compare-encoders,generate}
tiger: error: argument command: invalid choice: 'import-abo' (choose from synthgen, import-fashion, noise, calibrate, detect, evaluate, analyze, train-arbiter, route, repair, sweep, calibrate-fusion, ablate, ablate-repair, compare-encoders, generate)


## 2. Calibrate on ABO
This fits the new similarity thresholds for the non-fashion domain.

In [3]:
!python -m tiger.cli calibrate

locked thresholds -> /kaggle/working/TIGeR-Text-Image-Generative-Repair/data/thresholds/tiger_locked_thresholds.json
LOO calibration -> /kaggle/working/TIGeR-Text-Image-Generative-Repair/data/thresholds/tiger_loo_calibration.json
verify epsilon=0.0318 (by category: {'shirts': 0.0345, 'shoes': 0.034, 'bags': 0.0239, 'hats': 0.0265}) -> /kaggle/working/TIGeR-Text-Image-Generative-Repair/data/thresholds/tiger_verify_calibration.json


## 3. Inject Synthetic Noise
This injects errors into the report split so there is something to repair. **This step is required** — without it, the Arbiter and ablation have no corrupted products to work on.

In [ ]:
!python -m tiger.cli noise --seed 7

## 4. Retrain Arbiter
This retrains the Logistic Regression router on the new ABO-domain noise patterns.

In [4]:
!python -m tiger.cli train-arbiter

wrote /kaggle/working/TIGeR-Text-Image-Generative-Repair/data/processed/noisy_report_cal_seed1007.parquet
{
  "seed": 1007,
  "copies_per_row": 1,
  "rates": {
    "swap_image": 0.1,
    "swap_image_same_category": 0.03,
    "color_flip": 0.06,
    "near_color_flip": 0.02,
    "material_flip": 0.02,
    "attribute_drop": 0.02,
    "title_contradiction": 0.02,
    "mixed_swap_color": 0.02,
    "missing_image": 0.01
  },
  "rows_total": 120,
  "rows_noisy": 34,
  "by_label": {
    "clean": 86,
    "swap_image": 16,
    "mutate_text": 15,
    "mixed": 2,
    "missing_image": 1
  },
  "by_subtype": {
    "swap_image": 12,
    "color_flip": 7,
    "swap_image_same_category": 4,
    "material_flip": 2,
    "attribute_drop": 2,
    "near_color_flip": 2,
    "title_contradiction": 2,
    "mixed_swap_color": 2,
    "missing_image": 1
  },
  "self_verified": true
}


📊 Stage 1: Error Detection Complete
--------------------------------------------------
Out of 120 products scanned, we flagged 40 

## 5. Run Repair Ablation
This evaluates the repair pipeline using the Independent Verifier (SigLIP) and Generative Fallback.

In [5]:
!python -m tiger.cli ablate-repair --independent --generative-fallback

Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/kaggle/working/TIGeR-Text-Image-Generative-Repair/tiger/cli.py", line 726, in <module>
    main()
  File "/kaggle/working/TIGeR-Text-Image-Generative-Repair/tiger/cli.py", line 704, in main
    {
  File "/kaggle/working/TIGeR-Text-Image-Generative-Repair/tiger/cli.py", line 447, in cmd_ablate_repair
    noisy = pd.read_parquet(p["processed"] / f"noisy_report_{tag}.parquet")
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas/io/parquet.py", line 669, in read_parquet
    return impl.read(
           ^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas/io/parquet.py", line 258, in read
    path_or_handle, handles, filesystem = _get_path_or_handle(
                                          ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-pac

## 6. View Results
Compare this table with the Fashion results in your paper.

In [6]:
import pandas as pd
import glob
# Dynamically find whichever ablation CSV was produced
csvs = sorted(glob.glob('data/outputs/repair_ablations_summary*.csv'))
print('Found CSVs:', csvs)
if csvs:
    df = pd.read_csv(csvs[-1])
    print(df.to_string())
else:
    print('No results CSV found. Check that ablate-repair ran successfully.')